In [2]:
import re #удаление спецсимволы для LightGBM
import numpy as np
import pandas as pd
import lightgbm as lgb

from metric import precision_at_recall, recall_at_fpr

SEED = 42
np.random.seed(SEED)
N_VAL_DAYS = 6      # сколько последних дней трейна по очереди служат валидацией
N_BOOT = 300        # ресэмплов бутстрэпа для сравнения вариантов
FINAL_SEEDS = 5

# параметры v2; число деревьев = 411 (early stopping в v2) * 1.1
PARAMS = dict(objective="binary", learning_rate=0.02, num_leaves=15,
              min_child_samples=50, subsample=0.8, subsample_freq=1,
              colsample_bytree=0.8, n_estimators=453,
              random_state=SEED, deterministic=True, verbose=-1)

DATES = ["cookie_created_at", "window_start_ts", "window_end_ts"]

# грузим!
train = pd.read_csv("data/train.csv", parse_dates=DATES)
test = pd.read_csv("data/test.csv", parse_dates=DATES)
sample = pd.read_csv("sample_submission.csv")
events = pd.read_csv("data/events.csv.gz", parse_dates=["event_ts"])

In [3]:
# полные дубликаты событий не несут информации и искажают счётчики
dup_flag = events.duplicated()          # оставляем первое вхождение
dups = events.loc[dup_flag, ["cookie_id", "event_ts"]]
events = events[~dup_flag].copy()
events["row_idx"] = events.index              # исходный порядок строк в файле
print(f"Удалено дубликатов событий: {int(dup_flag.sum())}")

Удалено дубликатов событий: 4863


In [4]:
train["is_train"] = 1
test["is_train"] = 0
test["target"] = np.nan
meta = pd.concat([train, test], ignore_index=True)
meta["rid"] = np.arange(len(meta))  # id строки (кука + окно)

In [5]:
# только события внутри окна
# признаки должны быть доступны на момент конца окна: берём строго [start, end)
ev = events.merge(meta[["rid", "cookie_id", "window_start_ts", "window_end_ts"]], on="cookie_id")
ev = ev[(ev.event_ts >= ev.window_start_ts) & (ev.event_ts < ev.window_end_ts)]
ev = ev.sort_values(["rid", "event_ts"]).reset_index(drop=True)

g = ev.groupby("rid")
f = pd.DataFrame(index=meta.rid)

In [6]:
# объём и разнообразие
f["n_events"] = g.size()
for c in ["item_id", "item_category", "item_location", "seller_type",
          "search_query", "user_agent", "platform", "event_name"]:
    f[f"nuniq_{c}"] = g[c].nunique()
f["items_per_event"] = f.nuniq_item_id / f.n_events
f["search_page_max"] = g.search_page.max()
f["search_page_mean"] = g.search_page.mean()

In [7]:
# тайминги
ev["dt"] = g.event_ts.diff().dt.total_seconds()
d = ev.groupby("rid")["dt"]
f["dt_mean"] = d.mean()
f["dt_median"] = d.median()
f["dt_std"] = d.std()
f["dt_min"] = d.min()
f["dt_cv"] = f.dt_std / (f.dt_mean + 1e-9)          # ровность интервалов
f["dt_lt1s"] = (ev.dt < 1).groupby(ev.rid).mean()
f["dt_nuniq_ratio"] = ev.dt.round(1).groupby(ev.rid).nunique() / f.n_events

f["span_h"] = (g.event_ts.max() - g.event_ts.min()).dt.total_seconds() / 3600
f["events_per_hour"] = f.n_events / (f.span_h + 1 / 60)
ev["hour"] = ev.event_ts.dt.hour
f["n_active_hours"] = ev.groupby("rid").hour.nunique()
f["night_share"] = ev.hour.between(1, 5).groupby(ev.rid).mean()
per_min = ev.groupby(["rid", ev.event_ts.dt.floor("min")]).size()
f["max_per_min"] = per_min.groupby(level=0).max()

In [8]:
# курсор
f["pointer_na"] = ev.pointer_x.isna().groupby(ev.rid).mean()
f["pointer_x_std"] = g.pointer_x.std()
f["pointer_y_std"] = g.pointer_y.std()
pt = ev.dropna(subset=["pointer_x", "pointer_y"])
f["pointer_nuniq"] = (pt.pointer_x.astype(str) + "_" + pt.pointer_y.astype(str)).groupby(pt.rid).nunique()

In [9]:
# юзер агентик.!!
ua = ev.user_agent.fillna("").str.lower()
f["ua_na"] = ev.user_agent.isna().groupby(ev.rid).mean()
f["ua_len"] = ua.str.len().groupby(ev.rid).mean()
f["ua_suspicious"] = ua.str.contains(
    r"bot|crawl|spider|python|curl|wget|headless|selenium|phantom|scrapy|go-http|java"
).groupby(ev.rid).max().astype(float)
# сырую строку UA не берём (почти уникальна ->>>> модель запомнит источники)берём смысловые части
f["ua_chrome_ver"] = ua.str.extract(r"chrome/(\d+)")[0].astype(float).groupby(ev.rid).max()
f["ua_is_mobile"] = ua.str.contains(r"mobile|android|iphone").groupby(ev.rid).mean()

In [10]:
# доли типов событий и платформ капча, контакты, избранное
f = f.join(pd.crosstab(ev.rid, ev.event_name, normalize="index").add_prefix("share_"))
f = f.join(pd.crosstab(ev.rid, ev.platform.fillna("NA"), normalize="index").add_prefix("plat_"))

In [11]:
# последовательность действий
# человек ходит рвано: ищет, возвращается к объявлениям, листает фото а парсер идёт по шаблону
same_rid = ev.rid.eq(ev.rid.shift())
ev["same_event"] = (ev.event_name.eq(ev.event_name.shift()) & same_rid).astype(float)
f["share_repeat_event"] = ev.groupby("rid").same_event.mean()
ev["bigram"] = ev.event_name.shift().astype(str) + ">" + ev.event_name.astype(str)
ev.loc[~same_rid, "bigram"] = "START>" + ev.loc[~same_rid, "event_name"].astype(str)
f["nuniq_bigram_ratio"] = ev.groupby("rid").bigram.nunique() / f.n_events  # разнообразие переходов

iv = ev.dropna(subset=["item_id"])
f["item_revisit_share"] = 1 - iv.groupby("rid").item_id.nunique() / iv.groupby("rid").size()

sp = ev.dropna(subset=["search_page"])
seq_page = (sp.search_page.diff() == 1) & sp.rid.eq(sp.rid.shift())
f["search_page_seq_share"] = seq_page.groupby(sp.rid).mean()     # листает выдачу строго 1,2,3
f["query_len_mean"] = ev.search_query.dropna().astype(str).str.len().groupby(ev.rid).mean()

is_contact = ev.event_name.astype(str).str.contains("contact", case=False)
f["contacts_per_item"] = is_contact.groupby(ev.rid).sum() / (f.nuniq_item_id + 1)  # сбор контактов

In [12]:
# распределение интервалов подробнее
f["dt_p10"] = d.quantile(0.1)
f["dt_p90"] = d.quantile(0.9)
f["dt_iqr_ratio"] = (d.quantile(0.75) - d.quantile(0.25)) / (f.dt_median + 1e-9)

# курсор застыл или стоит в нуле
same_ptr = ev.pointer_x.eq(ev.pointer_x.shift()) & ev.pointer_y.eq(ev.pointer_y.shift()) & same_rid
f["pointer_same_share"] = same_ptr.groupby(ev.rid).mean()
f["pointer_zero_share"] = ((ev.pointer_x == 0) & (ev.pointer_y == 0)).groupby(ev.rid).mean()

# метаданные куки
m = meta.set_index("rid")
f["cookie_age_h"] = (m.window_start_ts - m.cookie_created_at).dt.total_seconds() / 3600
f["created_in_window"] = (m.cookie_created_at >= m.window_start_ts).astype(int)

# куки без событий в окне: счётчик = 0, остальные признаки NaN (LightGBM обрабатывает пропуски сам)
f["n_events"] = f["n_events"].fillna(0)
f.columns = [re.sub(r'[\[\]{}":,]', "_", str(c)) for c in f.columns]  # LightGBM не любит спецсимволы

In [13]:
# Новые признаки v3 (добавляются в конец, чтобы набор и порядок колонок v2 не изменились)
V2_COLS = list(f.columns)

# геометрия курсора
# У людей курсор почти никогда не стоит ровно на краю экрана (x=0, x>=1920, y=0, y>=1080),
# у ботов такие обрезанные координаты встречаются регулярно; кроме того, курсор ботов
# ходит в узкой зоне меньше максимум и среднее по x. Считаем только события с координатами.
px, py = ev.pointer_x, ev.pointer_y
edge = ((px <= 0) | (px >= 1920) | (py <= 0) | (py >= 1080)).astype(float).where(px.notna())
f["ptr_edge_share"] = edge.groupby(ev.rid).mean().astype(float)
f["ptr_x_max"] = px.groupby(ev.rid).max()
f["ptr_x_mean"] = px.groupby(ev.rid).mean()
f["ptr_y_max"] = py.groupby(ev.rid).max()
GROUP_PTR = ["ptr_edge_share", "ptr_x_max", "ptr_x_mean", "ptr_y_max"]

In [14]:
# толпа вокруг объявлений
# Парсеры обходят один и тот же пул объявлений, поэтому объявления, которые смотрит бот,
# часто смотрят и другие боты. Два признака, оба используют только прошлое:
#   ipop_share — какая доля всех кук (до конца текущего дня) уже смотрела эти объявления (без меток);
#   te_m/te_x  — доля ботов среди размеченных кук трейна, смотревших эти объявления РАНЬШЕ
#                (target encoding строго по прошлым дням, метки теста не используются).
# На тесте разметка отстаёт на 0–6 дней (тест — 7 дней после трейна). Чтобы модель не переоценивала
# свежесть разметки, на трейне отставание имитируется случайно 0–6 дней. te_lag — сам лаг.
TE_MAX_LAG = 6
TE_SMOOTH = 5
DAY_R = m.window_start_ts.dt.floor("D").astype("datetime64[ns]")
IS_TR_R = m.is_train.eq(1)

iv = ev.dropna(subset=["item_id"])[["rid", "item_id", "window_start_ts"]].copy()
iv["day"] = iv.window_start_ts.dt.floor("D").astype("datetime64[ns]")

cd = (iv.groupby(["item_id", "day"]).rid.nunique().rename("c").reset_index()
        .sort_values(["item_id", "day"]))
cd["cum"] = cd.groupby("item_id").c.cumsum()                 # кук, смотревших объявление, до конца дня
iv = iv.merge(cd[["item_id", "day", "cum"]], on=["item_id", "day"], how="left")
n_cookies_cum = DAY_R.value_counts().sort_index().cumsum()   # всего кук до конца дня
iv["pop_share"] = (iv["cum"] - 1) / iv["day"].map(n_cookies_cum)   # минус сама кука
f["ipop_share"] = (iv["pop_share"] * 1e4).groupby(iv.rid).mean()

In [15]:
rng_te = np.random.default_rng(SEED)
lag = pd.Series(rng_te.integers(0, TE_MAX_LAG + 1, len(m)), index=m.index)
first_test_day = DAY_R[~IS_TR_R].min()
cut = (DAY_R - pd.to_timedelta(lag, unit="D")).where(IS_TR_R, first_test_day).astype("datetime64[ns]")
f["te_lag"] = (DAY_R - cut).dt.days.astype(float)

lab = iv.join(m[["target"]], on="rid")
lab = lab[lab["target"].notna()].drop_duplicates(["item_id", "rid"])
hist = (lab.groupby(["item_id", "day"]).agg(v=("rid", "size"), b=("target", "sum")).reset_index()
          .sort_values(["item_id", "day"]))
hist["v_cum"] = hist.groupby("item_id").v.cumsum()
hist["b_cum"] = hist.groupby("item_id").b.cumsum()
hist = hist.rename(columns={"day": "cut"})[["item_id", "cut", "v_cum", "b_cum"]].sort_values("cut")

In [16]:

iv["cut"] = iv.rid.map(cut).astype("datetime64[ns]")
keys = iv[["item_id", "cut"]].drop_duplicates().sort_values("cut")
# allow_exact_matches=False: берём только дни СТРОГО раньше cut
keys = pd.merge_asof(keys, hist, on="cut", by="item_id", allow_exact_matches=False)
iv = iv.merge(keys, on=["item_id", "cut"], how="left")
iv[["v_cum", "b_cum"]] = iv[["v_cum", "b_cum"]].fillna(0)
prior = m["target"].mean()
iv["te"] = (iv["b_cum"] + TE_SMOOTH * prior) / (iv["v_cum"] + TE_SMOOTH)   # сглаживание к средней доле
f["te_m"] = iv.groupby("rid").te.mean()
f["te_x"] = iv.groupby("rid").te.max()
GROUP_CROWD = ["ipop_share", "te_m", "te_x", "te_lag"]


In [17]:
# страховка: все признаки приводим к float
f = f.astype(float)
# Надёжная валидация: несколько временных фолдов
# Теперь каждый из последних N дней по очереди — валидация, модель учится на всех более ранних днях
# Предсказания всех фолдов объединяются, метрика считается сразу на всех этих куках
# Сравнение вариантов - парный бутстрэп: доля ресэмплов, где вариант лучше эталона (REF)
is_tr = (m.is_train == 1).values
y = m.loc[is_tr, "target"].astype(int).values
days = m.loc[is_tr, "window_start_ts"].dt.floor("D").to_numpy()
uniq_days = np.unique(days)
n_val = min(N_VAL_DAYS, len(uniq_days) - 1)
assert n_val >= 2, "слишком мало дней в трейне для временных фолдов"
val_days = uniq_days[-n_val:]
in_val = np.isin(days, val_days)
yv = y[in_val]
print(f"Временные фолды: {n_val} последних дней, в валидации {in_val.sum()} кук, ботов {yv.sum()}")


def time_cv(cols, params):
    """Out-of-fold предсказания для кук из последних n_val дней."""
    X = f.loc[is_tr, cols]
    oof = np.full(len(y), np.nan)
    for d0 in val_days:
        trn, val = days < d0, days == d0
        model = lgb.LGBMClassifier(**params).fit(X[trn], y[trn])
        oof[val] = model.predict_proba(X[val])[:, 1]
    return oof[in_val]

Временные фолды: 6 последних дней, в валидации 4318 кук, ботов 355


In [18]:
rng = np.random.default_rng(SEED)
boot = [rng.integers(0, len(yv), len(yv)) for _ in range(N_BOOT)]

def boot_scores(p):
    return np.array([precision_at_recall(yv[b], p[b]) for b in boot])

# Прочие гипотезы что квартили, тайминги, UA, сессии, dwell, география, TE запросов/городов не помогли.
V3_COLS = V2_COLS + GROUP_PTR
REF = "v3 3 версия (тест 0.8261)"
VARIANTS = {
    REF:                (V3_COLS, PARAMS),
    "v4 = v3 + толпа":  (V3_COLS + GROUP_CROWD, PARAMS),
}
MIN_CONFIDENCE = 0.90

results = {}
ref_bs = None
for name, (cols, params) in VARIANTS.items():
    p = time_cv(cols, params)
    bs = boot_scores(p)
    if ref_bs is None:
        ref_bs = bs
        print(f"\nШум метрики на валидации: ±{bs.std():.4f} (1 std бутстрэпа)\n")
    delta = bs - ref_bs
    results[name] = (delta.mean(), (delta > 0).mean())
    print(f"{name:20s} P@R0.70 = {precision_at_recall(yv, p):.4f} | "
          f"R@FPR1% = {recall_at_fpr(yv, p):.4f} | "
          f"Δ к {REF.split()[0]} = {delta.mean():+.4f} | лучше в {(delta > 0).mean():.0%} ресэмплов")


Шум метрики на валидации: ±0.0431 (1 std бутстрэпа)

v3 3 версия (тест 0.8261) P@R0.70 = 0.7685 | R@FPR1% = 0.6338 | Δ к v3 = +0.0000 | лучше в 0% ресэмплов
v4 = v3 + толпа      P@R0.70 = 0.8834 | R@FPR1% = 0.7155 | Δ к v3 = +0.1037 | лучше в 100% ресэмплов


In [19]:

# выбираем вариант, который обгоняет эталон уверенно большее90% ресэмплов
confident = {k: v for k, v in results.items() if v[1] >= MIN_CONFIDENCE and k != REF}
best_name = max(confident, key=lambda k: confident[k][0]) if confident else REF
best_cols, best_params = VARIANTS[best_name]
print(f"\nФинальный вариант: {best_name} ({len(best_cols)} признаков)"
      + ("" if confident else " — новый вариант уверенно не лучше, сабмит совпадает с v3"))

# важности на выбранном наборе
imp_model = lgb.LGBMClassifier(**best_params).fit(f.loc[is_tr, best_cols], y)
imp = pd.Series(imp_model.booster_.feature_importance("gain"), index=best_cols)
print("\nТоп-20 признаков:\n", imp.sort_values(ascending=False).head(20))

# Финальная модель на всём трейне
X_all, X_test = f.loc[is_tr, best_cols], f.loc[~is_tr, best_cols]
pred = np.zeros(len(X_test))
for k in range(FINAL_SEEDS):
    model = lgb.LGBMClassifier(**{**best_params, "random_state": SEED + k}).fit(X_all, y)
    pred += model.predict_proba(X_test)[:, 1] / FINAL_SEEDS


Финальный вариант: v4 = v3 + толпа (74 признаков)

Топ-20 признаков:
 dt_median              15231.477792
ipop_share             12307.681304
pointer_x_std           8218.071121
ptr_edge_share          6717.206767
nuniq_item_location     6410.433465
te_m                    6203.941063
dt_iqr_ratio            3908.421396
nuniq_item_id           3366.518622
nuniq_item_category     2905.711092
dt_min                  2861.351246
dt_p90                  2767.314457
search_page_mean        2453.137007
dt_nuniq_ratio          2416.843697
n_events                2284.097970
search_page_max         1809.397366
ptr_x_max               1773.664373
contacts_per_item       1698.737714
dt_p10                  1688.885300
cookie_age_h            1629.975511
items_per_event         1555.240208
dtype: float64


In [20]:
#финаол
sub = pd.DataFrame({"cookie_id": m.loc[~is_tr, "cookie_id"].values, "score": pred})
sub = sample[["cookie_id"]].merge(sub, on="cookie_id", how="left")
assert sub.score.notna().all() and sub.cookie_id.is_unique and len(sub) == len(test), "формат сабмита нарушен"
sub.to_csv("submission.csv", index=False)
print("\nsubmission.csv сохранён:", sub.shape)


submission.csv сохранён: (4909, 2)
